In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# pip install torchvision
# 

In [3]:
import os,sys
#sys.path.append('/work/qdiff/mo_utils')
sys.executable


'/home/nadavg/phase2-sdk/sdk_virtualenv/bin/python'

In [4]:
print(os.getcwdb())
os.chdir('/home/nadavg/q-diffusion')
print(os.getcwdb())

b'/home/nadavg/q-diffusion/scripts/hf15'
b'/home/nadavg/q-diffusion'


In [5]:
os.environ['CUDA_VISIBLE_DEVICES'] = '4'

In [6]:
import torch
torch.cuda.is_available()

False

In [7]:
from mo_utils.utils.stand_alone_utils.har_utils import get_har_files,get_params_from_har
from mo_utils.utils.stand_alone_utils.pytorch2accelras import (
    get_nested_attr,
    get_weight_and_bias_from_layer_name,
    acc_ker_to_pytorch_weight,
    UpdateUnet,
    )
from mo_utils.utils.stand_alone_utils.quant_utils import calc_snr,calc_stats


In [8]:
from src.utils.torch_utils import add_full_name_to_module
from scripts.hf15.init_pipe import init_pipe

/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/flax/struct.py:136: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)
/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/flax/struct.py:136: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, keypaths)
2025-03-31 10:52:35.163555: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/flax/struct.py:136: FutureWarning: jax.tree_util.register_keypaths is deprecated, and will be removed in a future release. Please use `register_pytree_with_keys()` instead.
  jax.tree_util.register_keypaths(data_clz, k

In [9]:
import netron
from pathlib import Path
from collections import namedtuple

In [10]:
import torch
import torch.nn as nn
from diffusers import StableDiffusionPipeline
from scripts.hf15.txt2img import get_train_samples
from qdiff.quant_model import QuantModel

/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/transformers/models/clip/feature_extraction_clip.py:28: FutureWarning: The class CLIPFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use CLIPImageProcessor instead.
  warnings.warn(


In [11]:
cali_data_path ='/genai/users/nadavg/sd/qdiff_hf15_verb/gen_calib/calib_dict_steps20.pt' 
sample_data = torch.load(cali_data_path)
dummy_args = namedtuple('Args',['cali_n','cali_st','custom_steps','cond'])
opt = dummy_args(1,20,None,True)
cali_data = get_train_samples(opt, sample_data,20)
cali_xs, cali_ts, cali_cs = cali_data

In [12]:
pipe = init_pipe()
unet = pipe.unet
add_full_name_to_module(unet)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

/home/nadavg/phase2-sdk/sdk_virtualenv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, plea

In [13]:
pipe1 = init_pipe()
unet1 = pipe1.unet
add_full_name_to_module(unet1)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [14]:
id(unet),id(unet1)  

(135505082625056, 135504881533904)

In [15]:
tb =get_nested_attr(unet,'down_blocks.0.attentions.0.transformer_blocks.0',ignore_last=0,
                       spliter='.',)

In [16]:
type(tb)

diffusers.models.attention.BasicTransformerBlock

In [17]:
attn = get_nested_attr(unet,'down_blocks.0.attentions.0.transformer_blocks.0.attn1',ignore_last=0,
                       spliter='.',)

attn1 = get_nested_attr(unet1,'down_blocks.0.attentions.0.transformer_blocks.0.attn1',ignore_last=0,
                       spliter='.',)

In [18]:
ascale = attn.scale

In [19]:
ascale,ascale**0.5,(1/40)**0.5

(0.15811388300841897, 0.3976353643835253, 0.15811388300841897)

In [20]:
attn.query_dim, attn.heads,attn.query_dim/attn.heads

(320, 8, 40.0)

In [21]:
norm = get_nested_attr(unet,
                       'down_blocks.0.attentions.0.transformer_blocks.0.norm1',
                       ignore_last=0,spliter='.',)
norm1 = get_nested_attr(unet1,
                       'down_blocks.0.attentions.0.transformer_blocks.0.norm1',
                       ignore_last=0,spliter='.',)

In [22]:
from src.utils.torch_utils import save_input_hook

In [23]:
hook = norm.register_forward_hook(save_input_hook)

In [24]:
ind_batch= slice(30,31)
out_org = unet(cali_xs[ind_batch],cali_ts[ind_batch],cali_cs[ind_batch])
out_org[0].shape

torch.Size([1, 4, 64, 64])

In [25]:
norm.saved_inputs[0][0].shape

torch.Size([1, 4096, 320])

In [26]:
inp = norm.saved_inputs[0][0].detach().clone()

In [25]:
class normatt(nn.Module):
    def __init__(self,norm,attn):
        super().__init__()
        self.norm = norm
        self.attn = attn
    def forward(self,x):
        x = self.norm(x)
        x = self.attn(x)
        return x

In [26]:
#inp = torch.randn(1,4096,320)
na = normatt(norm,attn)
na1 = normatt(norm1,attn1)

In [27]:
out_org = na(inp)
out_org.shape

torch.Size([1, 4096, 320])

In [28]:
out_org1 = na1(inp)
calc_snr(out_org1,out_org)

139.37945161183634

In [49]:
from mo_utils.utils.har_utils import super_netron

In [51]:
super_netron('/home/nadavg/q-diffusion/normattn.sim.onnx')

Serving '/home/nadavg/q-diffusion/normattn.sim.onnx' at http://localhost:8081


In [51]:
super_netron('normattn_sim.har')

all_names=['normattn_sim.hn', 'normattn_sim.npz', 'normattn_sim.original_model_meta.json', 'normattn_sim.metadata.json']
loading  params_name='normattn_sim.hn' ...
Serving 'temp.hn' at http://localhost:21403


In [26]:
from hailo_sdk_client.runner.client_runner import ClientRunner

In [31]:
if False:
    cr = ClientRunner(har='normattn_sim.har',)
    cr.optimize_full_precision(calib_data=inp)
    cr.save_har('normattn_sim_fp.har')

In [29]:
from mo_utils.utils.acceleras_utils_verb import AccModel 
from hailo_sdk_client import InferenceContext


ac_na = AccModel(har_path='normattn_sim_fp.har',inf_context=InferenceContext.SDK_FP_OPTIMIZED)

warning! no input shape sp model mot build. 
 run inference before using the model


2025-03-31 00:12:12.559434: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [30]:
inp.shape
#inp.unsqueeze(0).shape

torch.Size([1, 4096, 320])

In [31]:
out_ac = ac_na.model(inp.unsqueeze(0).numpy())

In [32]:
inter_res = ac_na.get_inter_reses_full(
    model=ac_na.model,
    input=inp.unsqueeze(0).numpy(),
    only_native_res=True
    )

In [33]:
inter_res['native']['normattn_sim/layer_normalization1'][0].shape

TensorShape([1, 1, 4096, 320])

In [34]:
calc_snr(out_ac[0],out_org[0])

123.74845118124796

In [57]:
super_netron('normattn_sim_fp.har')

all_names=['normattn_sim.hn', 'normattn_sim.native.hn', 'normattn_sim.npz', 'normattn_sim.fpo.npz', 'normattn_sim.stats.npz', 'normattn_sim.original_model_meta.json', 'normattn_sim.modifications_meta_data.json', 'normattn_sim.modification_params.npz', 'normattn_sim.metadata.json']
loading  params_name='normattn_sim.hn' ...
Serving 'temp.hn' at http://localhost:24296


In [35]:
ace_unet_path =  'normattn_sim_fp.har'
get_har_files(ace_unet_path)


['normattn_sim.hn',
 'normattn_sim.native.hn',
 'normattn_sim.npz',
 'normattn_sim.fpo.npz',
 'normattn_sim.stats.npz',
 'normattn_sim.original_model_meta.json',
 'normattn_sim.modifications_meta_data.json',
 'normattn_sim.modification_params.npz',
 'normattn_sim.metadata.json']

In [36]:
params = get_params_from_har(ace_unet_path,params_name='normattn_sim.fpo.npz')
hn = get_params_from_har(ace_unet_path,params_name='normattn_sim.hn')

all_names=['normattn_sim.hn', 'normattn_sim.native.hn', 'normattn_sim.npz', 'normattn_sim.fpo.npz', 'normattn_sim.stats.npz', 'normattn_sim.original_model_meta.json', 'normattn_sim.modifications_meta_data.json', 'normattn_sim.modification_params.npz', 'normattn_sim.metadata.json']
loading  params_name='normattn_sim.fpo.npz' ...
all_names=['normattn_sim.hn', 'normattn_sim.native.hn', 'normattn_sim.npz', 'normattn_sim.fpo.npz', 'normattn_sim.stats.npz', 'normattn_sim.original_model_meta.json', 'normattn_sim.modifications_meta_data.json', 'normattn_sim.modification_params.npz', 'normattn_sim.metadata.json']
loading  params_name='normattn_sim.hn' ...


In [ ]:
if False:
    native_unet_path =  'normattn_sim.har'
    get_har_files(native_unet_path)

    params_native = get_params_from_har(native_unet_path,params_name='normattn_sim.npz')
    hn = get_params_from_har(native_unet_path,params_name='normattn_sim.hn')
    ker,bias,kk,bb= get_weight_and_bias_from_layer_name('normattn_sim/conv1',params_native)
    ker.shape,bias.shape
    kn,bn,_,_=get_weight_and_bias_from_layer_name('normattn_sim/normalization1',params_native)
    kn.shape,bn.shape
    calc_snr(norm.weight, kn[0,0,:,0]),calc_snr(norm.bias, bn)

['normattn_sim.hn',
 'normattn_sim.npz',
 'normattn_sim.original_model_meta.json',
 'normattn_sim.metadata.json']

In [37]:
uu = UpdateUnet(na,hn,params)

In [38]:
uu.debug = True
uu.update_conv_weights_and_biases('normattn_sim/conv_feature_splitter1_1')
uu.update_conv_weights_and_biases('normattn_sim/conv_feature_splitter1_2')
uu.update_conv_weights_and_biases('normattn_sim/conv_feature_splitter1_3')

layer_name='normattn_sim/conv_feature_splitter1_1' : layer_norm_input=True,ff_input=False,kqv=True,norm.full_name='norm'
attn.full_name='attn',attn.scale=1.0
layer_name='normattn_sim/conv_feature_splitter1_2' : layer_norm_input=True,ff_input=False,kqv=True,norm.full_name='norm'
attn.full_name='attn',attn.scale=1.0
layer_name='normattn_sim/conv_feature_splitter1_3' : layer_norm_input=True,ff_input=False,kqv=True,norm.full_name='norm'
attn.full_name='attn',attn.scale=1.0


In [39]:
out_norm = na.norm(inp)

In [40]:
calc_snr(out_norm,inter_res['native']['normattn_sim/layer_normalization1'][0][0])

144.60699120245852

In [41]:
q= na.attn.to_q(out_norm)
k= na.attn.to_k(out_norm)
v= na.attn.to_v(out_norm)     
q.shape,k.shape,v.shape

(torch.Size([1, 4096, 320]),
 torch.Size([1, 4096, 320]),
 torch.Size([1, 4096, 320]))

In [43]:
(calc_snr(q,inter_res['native']['normattn_sim/conv_feature_splitter1_1'][0][0]),
calc_snr(k,inter_res['native']['normattn_sim/conv_feature_splitter1_2'][0][0]),
calc_snr(v,inter_res['native']['normattn_sim/conv_feature_splitter1_3'][0][0]))

(129.97124698155437, 129.17504434711415, 129.8745525650097)

In [44]:
out_norm1 = na1.norm(inp)
q1= na1.attn.to_q(out_norm1)
k1= na1.attn.to_k(out_norm1)
v1= na1.attn.to_v(out_norm1)     
q1.shape,k1.shape,v1.shape

(torch.Size([1, 4096, 320]),
 torch.Size([1, 4096, 320]),
 torch.Size([1, 4096, 320]))

In [47]:
np.sqrt(ascale)

0.3976353643835253

In [45]:
import numpy as np
#calc_snr(q1/np.sqrt(ascale),q),calc_snr(k1,k),calc_snr(v1,v)
calc_snr(q1*np.sqrt(ascale),q),calc_snr(k1*np.sqrt(ascale),k),calc_snr(v1,v)

(130.72062812710948, 130.2550778287334, 131.33936404671456)

In [46]:
na1.attn.scale,na.attn.scale

(0.15811388300841897, 1.0)

In [68]:
from einops import rearrange, repeat
h = na.attn.heads
q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> (b h) n d', h=h), (q, k, v))
q.shape,k.shape,v.shape

(torch.Size([8, 4096, 40]),
 torch.Size([8, 4096, 40]),
 torch.Size([8, 4096, 40]))

In [69]:
from torch import einsum

sim = einsum('b i d, b j d -> b i j', q, k) #* self.scale

In [70]:
sim.shape

torch.Size([8, 4096, 4096])

In [72]:
inter_res['native']['normattn_sim/matmul1'][0].shape

TensorShape([1, 1, 4096, 32768])

In [73]:
calc_snr(sim[0],inter_res['native']['normattn_sim/matmul1'][0][0,0,:,:4096])

0.38607200116638013

In [86]:
mm0 = torch.matmul(q[-1], k[-1].transpose(1, 0))
mm0.shape

torch.Size([4096, 4096])

In [87]:
calc_snr(mm0,inter_res['native']['normattn_sim/matmul1'][0][0,0,:,:4096])

-3.8088091702963656

In [75]:
sim1 = torch.matmul(q, k.transpose(-2, -1))
sim1.shape

torch.Size([8, 4096, 4096])

In [88]:
calc_snr(sim1,sim),calc_snr(sim1[-1],mm0)

(164.67311536506344, 160.72921155754108)

In [47]:
import torch.nn.functional as F
inner_dim = k.shape[-1]
head_dim = inner_dim // attn.heads
inner_dim,head_dim

(320, 40)

In [48]:
ascale,head_dim,(head_dim)**-0.5


(0.15811388300841897, 40, 0.15811388300841897)

In [88]:
key.shape

torch.Size([1, 8, 4096, 40])

In [55]:
query = q.view(1, -1, attn.heads, head_dim).transpose(1, 2)

key = k.view(1, -1, attn.heads, head_dim).transpose(1, 2)
value = v.view(1, -1, attn.heads, head_dim).transpose(1, 2)



        # the output of sdp = (batch, num_heads, seq_len, head_dim)
        # TODO: add support for attn.scale when we move to Torch 2.1
hidden_states = F.scaled_dot_product_attention(
    query, key*ascale, value, attn_mask=None, dropout_p=0.0, is_causal=False
)

hidden_states = hidden_states.transpose(1, 2).reshape(1, -1, attn.heads * head_dim)
hidden_states = hidden_states.to(q.dtype)

In [56]:
calc_snr(hidden_states,inter_res['native']['normattn_sim/matmul2'][0][0])

-2.622354373642208

In [96]:
attn.norm_q, attn.norm_k ,attn.residual_connection, attn.rescale_output_factor

(None, None, False, 1.0)

In [68]:
calc_snr(na(inp),out_org)

-1.4558810326156524

In [67]:
import torch.nn.functional as F

In [ ]:
calc_snr(na.out)

In [228]:
ker,bias,kk,bb= get_weight_and_bias_from_layer_name('normattn_sim/conv2',params_native)
ker.shape,bias.shape

((1, 1, 320, 320), (320,))

In [227]:
na.attn.to_out[0].weight.shape

torch.Size([320, 320])

In [230]:
calc_snr(ker,na.attn.to_out[0].weight.data.numpy().transpose(1,0))

130.06899308201375

In [231]:
calc_snr(bias,na.attn.to_out[0].bias.data.numpy())

129.92092252104203

In [138]:
na.in_channels = 320

In [54]:
na.norm.weight

In [140]:
out_reorg_unet = na(inp)

In [49]:
from types import MethodType
from qdiff.quant_block import cross_attn_forward
na.attn.forward = MethodType(cross_attn_forward, na.attn)
na.attn.to_out = nn.Sequential(na.attn.to_out[0],na.attn.to_out[1])
na.attn.use_act_quant = False
na.attn.use_weight_quant = False
na.attn.scale

1.0

In [50]:
out_reorg_qnn = na(inp)


In [51]:
calc_snr(out_org,out_reorg_qnn)#,calc_snr(out_org[0],out_reorg_unet[0])

126.53851621754079

In [115]:
qnn.model

normatt(
  (norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
  (attn): Attention(
    (to_q): QuantModule(
      in_features=320, out_features=320, bias=True
      (weight_quantizer): UniformAffineQuantizer(bit=8, scale_method=max, symmetric=False, channel_wise=False, leaf_param=False)
      (act_quantizer): UniformAffineQuantizer(bit=8, scale_method=max, symmetric=False, channel_wise=False, leaf_param=False)
      (activation_function): StraightThrough()
    )
    (to_k): QuantModule(
      in_features=320, out_features=320, bias=True
      (weight_quantizer): UniformAffineQuantizer(bit=8, scale_method=max, symmetric=False, channel_wise=False, leaf_param=False)
      (act_quantizer): UniformAffineQuantizer(bit=8, scale_method=max, symmetric=False, channel_wise=False, leaf_param=False)
      (activation_function): StraightThrough()
    )
    (to_v): QuantModule(
      in_features=320, out_features=320, bias=True
      (weight_quantizer): UniformAffineQuantizer(bit=8, scale_

In [ ]:
?torch.onnx.export

Signature:
torch.onnx.export(
    model: 'torch.nn.Module | torch.export.ExportedProgram | torch.jit.ScriptModule | torch.jit.ScriptFunction',
    args: 'tuple[Any, ...]' = (),
    f: 'str | os.PathLike | None' = None,
    *,
    kwargs: 'dict[str, Any] | None' = None,
    export_params: 'bool' = True,
    verbose: 'bool | None' = None,
    input_names: 'Sequence[str] | None' = None,
    output_names: 'Sequence[str] | None' = None,
    opset_version: 'int | None' = None,
    dynamic_axes: 'Mapping[str, Mapping[int, str]] | Mapping[str, Sequence[int]] | None' = None,
    keep_initializers_as_inputs: 'bool' = False,
    dynamo: 'bool' = False,
    external_data: 'bool' = True,
    dynamic_shapes: 'dict[str, Any] | tuple[Any, ...] | list[Any] | None' = None,
    custom_translation_table: 'dict[Callable, Callable | Sequence[Callable]] | None' = None,
    report: 'bool' = False,
    optimize: 'bool' = False,
    verify: 'bool' = False,
    profile: 'bool' = False,
    dump_exported_program: